In [1]:
import geemap
import ee
import numpy as np
import pandas as pd
ee.Authenticate()
ee.Initialize(project='ee-ivanburgov666')


In [ ]:
greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

# create a vector of month time steps
months = np.arange(1, 13, 1) # last month not inclusive! # 1,13,1
date_start_initial = ee.Date('2002-01-01')

for s in months:
    adv=int(s-1)
    date_start = ee.Date(date_start_initial.advance(adv, 'month'))
    date_end = ee.Date(date_start.advance(1, 'month')) # month
    print(f'Processing month {s}')

    era5_col = (
    ee.ImageCollection("ECMWF/ERA5/HOURLY")
    .filterDate(date_start, date_end)
    .select(["total_cloud_cover"])
    .filterBounds(greenland)
    .mean()
    .set("month", s)
)

# Create monthly mean cloud cover
def monthly_mean_cloud_cover(month):
    start = ee.Date.fromYMD(2025, month, 1)
    end = start.advance(1, "month")
    monthly_mean = (
        era5_col.filterDate(start, end).mean().set("month", month)
    )
    return monthly_mean

# Loop through each month and create a list of monthly mean images
monthly_mean_images = [monthly_mean_cloud_cover(month) for month in range(1, 13)]

monthly_mean_images_collection = ee.ImageCollection.fromImages(monthly_mean_images)

monthly_mean_images_collection = monthly_mean_images_collection.map(
    lambda image: image.clip(greenland)
)
monthly_mean_images_collection


Processing month 1: from 2002-01-01 to 2002-02-01
Processing month 2: from 2002-02-01 to 2002-03-01
Processing month 3: from 2002-03-01 to 2002-04-01
Processing month 4: from 2002-04-01 to 2002-05-01
Processing month 5: from 2002-05-01 to 2002-06-01
Processing month 6: from 2002-06-01 to 2002-07-01
Processing month 7: from 2002-07-01 to 2002-08-01
Processing month 8: from 2002-08-01 to 2002-09-01
Processing month 9: from 2002-09-01 to 2002-10-01
Processing month 10: from 2002-10-01 to 2002-11-01
Processing month 11: from 2002-11-01 to 2002-12-01
Processing month 12: from 2002-12-01 to 2003-01-01
[ 1  2  3  4  5  6  7  8  9 10 11 12]


In [11]:
img = monthly_mean_images_collection.first()

Map = geemap.Map()
Map.addLayer(img, {"min": 0, "max": 1, "palette": ["blue", "white"]}, "Monthly Mean Cloud Cover")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…